In [131]:
import pandas as pd

In [220]:
df_adrs = pd.read_csv('./adrs.csv')
df_biomarkers = pd.read_csv('./biomarkers.csv')
df_subjects = pd.read_csv('./subjects.csv')

In [222]:
df_adrs.head()

,STUDYID,USUBJID,SUBJID,COLPROT,PARAMCD,PARAM,PARAMN,AVISIT,AVISITN,VISIT,...,AGE,AGEU,AGEGR1,AGEGR1N,SEX,RACE,BMI,BMIBLU,DX,AMYSTAT
0,ADNI,ADNI-001-00221,221,ADNI1,DX,Clinical Diagnostics Status,1,ADNI1 Month 12,370.0,ADNI1 Month 12,...,67.5,Years,>=65,2.0,Male,White,NaN,NaN,DEM,NaN
1,ADNI,ADNI-001-00221,221,ADNI1,DX,Clinical Diagnostics Status,1,ADNI1 Month 24,757.0,ADNI1 Month 24,...,67.5,Years,>=65,2.0,Male,White,NaN,NaN,DEM,NaN
2,ADNI,ADNI-001-00221,221,ADNI1,DX,Clinical Diagnostics Status,1,ADNI1 Month 6,181.0,ADNI1 Month 6,...,67.5,Years,>=65,2.0,Male,White,NaN,NaN,DEM,NaN
3,ADNI,ADNI-001-00221,221,ADNI1,DX,Clinical Diagnostics Status,1,BASELINE,0.0,ADNI1 Baseline,...,67.5,Years,>=65,2.0,Male,White,NaN,NaN,DEM,NaN
4,ADNI,ADNI-001-00222,222,ADNI1,DX,Clinical Diagnostics Status,1,ADNI1 Month 12,377.0,ADNI1 Month 12,...,85.9,Years,>=65,2.0,Male,White,NaN,NaN,MCI,NaN


In [224]:
# source: https://adni-lde.loni.usc.edu/quick-start-guide-asset/anatomy2.html?utm_source=chatgpt.com
mapping = {
    # ADNI GO
    ('ADNIGO', 'bl'): 'ADNIGO Baseline',
    ('ADNIGO', 'm06'): 'ADNIGO Month 6',
    ('ADNIGO', 'm48'): 'ADNIGO Month 12',
    ('ADNIGO', 'm60'): 'ADNIGO Month 60',
    ('ADNIGO', 'm72'): 'ADNIGO Month 72',
    ('ADNIGO', 'm12'): 'ADNIGO Month 12',
    ('ADNIGO', 'm36'): 'ADNIGO Month 36',
    
    # ADNI 1
    ('ADNI1', 'm12'): 'ADNI1 Month 12',
    ('ADNI1', 'm24'): 'ADNI1 Month 24',
    ('ADNI1', 'm06'): 'ADNI1 Month 6',
    ('ADNI1', 'bl'): 'ADNI1 Baseline',
    ('ADNI1', 'm18'): 'ADNI1 Month 18',
    ('ADNI1', 'm36'): 'ADNI1 Month 36',
    ('ADNI1', 'm48'): 'ADNI1 Month 48',
    ('ADNI1', 'uns1'): 'ADNI1 Unscheduled',

    # ADNI 2
    ('ADNI2', 'v06'): 'ADNI2 Initial Visit - Continuing Pt',
    ('ADNI2', 'v11'): 'ADNI2 Year 1 Visit',
    ('ADNI2', 'v21'): 'ADNI2 Year 2 Visit',
    ('ADNI2', 'v31'): 'ADNI2 Year 3 Visit',
    ('ADNI2', 'v41'): 'ADNI2 Year 4 Visit',
    ('ADNI2', 'v51'): 'ADNI2 Year 5 Visit',
    ('ADNI2', 'v05'): 'ADNI2 Month 6 - New Pt',
    ('ADNI2', 'v01'): 'ADNI2 Screening - New Pt',
    ('ADNI2', 'v03'): 'ADNI2 Baseline - New Pt',

    # ADNI 3
    ('ADNI3', 'init'): 'ADNI3 Initial Visit - Continuing Pt',
    ('ADNI3', 'y1'): 'ADNI3 Year 1 Visit',
    ('ADNI3', 'y2'): 'ADNI3 Year 2 Visit',
    ('ADNI3', 'sc'): 'ADNI3 Screening - New Pt',
    ('ADNI3', 'bl'): 'ADNI3 Baseline - New Pt',
    ('ADNI3', 'y4'): 'ADNI3 Year 4 Visit',
    ('ADNI3', 'y3'): 'ADNI3 Year 3 Visit',
    ('ADNI3', 'y5'): 'ADNI3 Year 5 Visit',
    
    # ADNI 4
    ('ADNI4', '4_init'): 'ADNI4 Initial Visit - Continuing Pt',
    ('ADNI4', '4_m12'): 'ADNI4 Month 12',
    ('ADNI4', '4_sc'): 'ADNI4 Screening - New Pt',
    ('ADNI4', '4_bl'): 'ADNI4 Baseline - New Pt',
    ('ADNI4', '4_m24'): 'ADNI4 Month 24',
}

In [226]:
df_biomarkers['VISIT'] = list(zip(df_biomarkers['COLPROT'], df_biomarkers['VISCODE']))
df_biomarkers['VISIT'] = df_biomarkers['VISIT'].map(mapping)

In [228]:
df_adrs['RID'] = df_adrs['SUBJID']
df_subjects['RID'] = df_subjects['SUBJID']

In [230]:
df_merged = df_adrs.merge(
    df_biomarkers,
    on=['RID', 'VISIT'],
    how='left'
)

df_merged = df_merged.merge(
    df_subjects,
    on=['RID'],
    how='left'
)

In [ ]:
for col in df_merged.columns:
    if col.endswith('_x'):
        base = col[:-2]
        col_x = col
        col_y = base + '_y'
        if col_y in df_merged.columns:
            df_merged[base] = df_merged[col_x].combine_first(df_merged[col_y])

df_merged = df_merged.drop(columns=[c for c in df_merged.columns if c.endswith('_x') or c.endswith('_y')])

In [234]:
df_merged.to_csv('master.csv', index=False)